# Weight Initialization: where training starts matters

We reuse the plain MLP + Fashion-MNIST training loop (no convolutions) and change **only how the weights are initialized** before training starts.

**Plan**
1. Look at what PyTorch does by default
2. See, *without training*, how a signal travels through a deep network under different inits
3. **Predict first**, then train the same MLP from 7 different initializations
4. Read the results: stuck, symmetric, exploding, fine
5. Prove the "symmetry" problem by looking at the trained weights

## 0. Setup

In [ ]:
import math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

## 1. Data and model

In [ ]:
mean, std = 0.2860, 0.3530
tr = datasets.FashionMNIST('data', train=True,  download=True)
te = datasets.FashionMNIST('data', train=False, download=True)

def to_tensors(ds):
    x = (ds.data.float() / 255 - mean) / std
    return x.unsqueeze(1), ds.targets

Xtr, ytr = to_tensors(tr)
Xte, yte = to_tensors(te)

train_ds          = TensorDataset(Xtr, ytr)
train_eval_loader = DataLoader(train_ds, batch_size=1000)
test_loader       = DataLoader(TensorDataset(Xte, yte), batch_size=1000)

class MLP(nn.Module):
    def __init__(self, act=nn.ReLU, hidden=(128, 64)):
        super().__init__()
        layers, d = [nn.Flatten()], 784
        for h in hidden:
            layers += [nn.Linear(d, h), act()]
            d = h
        layers.append(nn.Linear(d, 10))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

criterion = nn.CrossEntropyLoss()
print(MLP())

## 2. What does PyTorch do by default?

Every `nn.Linear` initializes itself when it is created. Let's look at the numbers. The key quantity is **fan_in**, the number of inputs to a layer: a neuron adds up `fan_in` products, so the *spread* of the weights should shrink as `fan_in` grows to keep the sum from blowing up.

In [ ]:
torch.manual_seed(0)
m = MLP()
for name, p in m.named_parameters():
    fan_in = p.shape[1] if p.ndim == 2 else None
    extra = f'fan_in={fan_in:4d}  1/sqrt(fan_in)={1/math.sqrt(fan_in):.4f}' if fan_in else ''
    print(f'{name:10s} shape={tuple(p.shape)!s:11s} mean={p.mean().item(): .4f}  std={p.std().item():.4f}  {extra}')

The default weights are small random numbers whose spread scales like `1/sqrt(fan_in)` (the std of the first layer, with fan_in=784, is much smaller than that of the last, with fan_in=64). So the default is *already* a "scaled" initialization. The experiments below show what happens when we throw that scaling away.

## 3. The initialization strategies

We define each one as a function that is applied to every `nn.Linear` layer with `model.apply(fn)`. Biases are set to zero (the usual choice), except for the PyTorch default, which we leave untouched.

| Strategy | Weights | Idea |
|---|---|---|
| zeros | all 0 | the "obvious" choice |
| constant | all 0.05 | same nonzero value everywhere |
| tiny random | Normal(0, 0.001) | random but very small |
| huge random | Normal(0, 1.0) | random but very large |
| PyTorch default | Uniform(+-1/sqrt(fan_in)) (Kaiming-uniform) | built-in |
| Xavier (Glorot) | variance 2 / (fan_in + fan_out) | designed for tanh / sigmoid |
| Kaiming (He) | variance 2 / fan_in | designed for ReLU |

In [ ]:
def _linear_only(fn):
    def wrapped(module):
        if isinstance(module, nn.Linear):
            fn(module)
    return wrapped

@_linear_only
def init_zeros(m):
    nn.init.zeros_(m.weight); nn.init.zeros_(m.bias)

@_linear_only
def init_constant(m):
    nn.init.constant_(m.weight, 0.05); nn.init.zeros_(m.bias)

def init_normal(std):
    @_linear_only
    def fn(m):
        nn.init.normal_(m.weight, mean=0.0, std=std); nn.init.zeros_(m.bias)
    return fn

@_linear_only
def init_xavier(m):
    nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

@_linear_only
def init_kaiming(m):
    nn.init.kaiming_normal_(m.weight, nonlinearity='relu'); nn.init.zeros_(m.bias)

INITS = {
    'zeros':            init_zeros,
    'constant 0.05':    init_constant,
    'tiny (std 0.001)': init_normal(0.001),
    'huge (std 1.0)':   init_normal(1.0),
    'PyTorch default':  None,                  # leave PyTorch's own init alone
    'Xavier':           init_xavier,
    'Kaiming (He)':     init_kaiming,
}

def make_model(init_fn, seed=0, **kw):
    torch.manual_seed(seed)
    model = MLP(**kw)
    if init_fn is not None:
        model.apply(init_fn)
    return model.to(device)

## 4. See the problem *without training*: signal flow through a deep network

Before any training, push one batch through a **deep** ReLU network (10 hidden layers, width 256) and record, layer by layer:
- the spread (std) of the activations going **forward**, and
- the size of the gradient going **backward**.

A healthy init keeps both roughly constant from layer to layer. A bad one makes them shrink toward 0 (vanishing) or grow without bound (exploding). The effect is much easier to see in a deep net, so we use one here just for this demo.

In [ ]:
def signal_stats(init_fn, depth=10, width=256, seed=0):
    model = make_model(init_fn, seed=seed, hidden=(width,) * depth)
    acts = []
    for mod in model.net:
        if isinstance(mod, nn.ReLU):
            mod.register_forward_hook(lambda mod, inp, out: acts.append(out.detach().std().item()))
    x, y = Xtr[:256].to(device), ytr[:256].to(device)
    model.zero_grad()
    criterion(model(x), y).backward()
    grads = [l.weight.grad.norm().item() for l in model.net if isinstance(l, nn.Linear)]
    return acts, grads

demo_inits = {'std 0.01': init_normal(0.01), 'std 1.0': init_normal(1.0), 'PyTorch default': None,
              'Xavier': init_xavier, 'Kaiming (He)': init_kaiming}

fig, ax = plt.subplots(1, 2, figsize=(14, 4.3))
for name, fn in demo_inits.items():
    acts, grads = signal_stats(fn)
    ax[0].plot(range(1, len(acts) + 1), acts, marker='o', ms=3, label=name)
    ax[1].plot(range(1, len(grads) + 1), grads, marker='o', ms=3, label=name)
ax[0].set_title('forward: std of activations after each hidden layer')
ax[0].set_xlabel('hidden layer (1 = nearest the input)')
ax[1].set_title('backward: gradient norm of each layer\'s weights')
ax[1].set_xlabel('layer (1 = nearest the input, 11 = output)')
for a in ax: a.set_yscale('log')
ax[0].legend(); plt.tight_layout(); plt.show()

How to read this (log scale, so each gridline is 10x):
- **std 0.01 (too small):** activations shrink layer after layer toward zero, so there is almost no signal for the output layer to use, and gradients are tiny too.
- **std 1.0 (too large):** activations grow by a large factor per layer and end up astronomically large.
- **Kaiming (He):** designed so that, for ReLU, the activation size stays about the same from layer to layer.
- **Xavier** is built for tanh/sigmoid, so with ReLU it still shrinks a bit each layer (it ignores that ReLU zeroes half the values).
- **PyTorch default** is tuned to be safe for shallow networks. In a 10-layer ReLU network it shrinks noticeably, which is one reason very deep nets need a careful init (and normalization layers).

## 5. Training function

The same 5-step loop as before, wrapped in `run(init_fn)`. Two details make the comparison fair:
- the starting seed is the same, and
- the **batch order is identical** in every run (the `DataLoader` gets its own random generator), so the only difference between runs is the initialization.

Training stops early if the loss becomes NaN or infinite.

In [ ]:
def evaluate(model, loader):
    model.eval()
    total, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            total += criterion(out, y).item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total / n, correct / n

def run(init_fn, epochs=10, lr=0.01, momentum=0.9, seed=0, verbose=True, name=''):
    model = make_model(init_fn, seed=seed)
    g = torch.Generator().manual_seed(seed)                      # same batch order for every run
    loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=g)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    h = dict(step_loss=[], train_loss=[], test_loss=[], test_acc=[], diverged=False, model=model)

    for epoch in range(1, epochs + 1):
        model.train()
        running, n = 0.0, 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()                 # 1. clear gradients
            loss = criterion(model(x), y)         # 2-3. forward + loss
            loss.backward()                       # 4. backward
            optimizer.step()                      # 5. update
            l = loss.item()
            h['step_loss'].append(l)
            if not math.isfinite(l):
                h['diverged'] = True
                break
            running += l * x.size(0); n += x.size(0)
        if h['diverged']:
            if verbose: print(f'  {name}: DIVERGED (loss became NaN/inf) in epoch {epoch}')
            break
        tl, ta = evaluate(model, test_loader)
        h['train_loss'].append(running / n); h['test_loss'].append(tl); h['test_acc'].append(ta)
        if verbose: print(f'  {name}: epoch {epoch:2d}  train_loss={running/n:.4f}  test_loss={tl:.4f}  test_acc={ta:.4f}')
    return h

## 6. Predict first

Setting: plain MLP `784 -> 128 -> 64 -> 10` with ReLU, SGD with lr 0.01 and momentum 0.9, 10 epochs. Chance level is 10% accuracy, loss `ln 10 = 2.30`.

| Init | Initial loss (guess) | Will it learn? | Test accuracy after 10 epochs (guess) |
|---|---|---|---|
| zeros | | | |
| constant 0.05 | | | |
| tiny (std 0.001) | | | |
| huge (std 1.0) | | | |
| PyTorch default | | | |
| Xavier | | | |
| Kaiming (He) | | | |

## 7. Step 1: measure each init *before* training

With 10 balanced classes, a network that knows nothing should start near loss `ln(10) = 2.30`. A starting loss far from that is a warning sign. We also measure the spread of the first hidden layer's pre-activations.

In [ ]:
xb, yb = Xtr[:2000].to(device), ytr[:2000].to(device)
rows = []
for name, fn in INITS.items():
    model = make_model(fn)
    model.eval()
    with torch.no_grad():
        loss = criterion(model(xb), yb).item()
        pre1 = model.net[1](xb.flatten(1)).std().item()
    rows.append({'init': name, 'initial_loss': loss, 'first-layer pre-activation std': pre1})
display(pd.DataFrame(rows).set_index('init').round(4))
print('guessing level: ln(10) =', round(math.log(10), 4))

Before any training:
- the **huge** init starts at an enormous loss, because its pre-activations are very large, so the logits are huge and wrong;
- **zeros** and **tiny** start at (almost exactly) 2.30, but for the wrong reason: their logits are all about 0, so the network predicts a flat 10% for every class.

## 8. Step 2: train from each initialization

In [ ]:
EPOCHS = 10
results = {}
for name, fn in INITS.items():
    t0 = time.time()
    results[name] = run(fn, epochs=EPOCHS, verbose=False, name=name)
    r = results[name]
    status = 'DIVERGED' if r['diverged'] else f"test_acc={r['test_acc'][-1]:.4f}"
    print(f'{name:18s} {status:18s} ({time.time()-t0:.0f}s)')

### 8a. Per-step loss (first 500 steps)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
for name, r in results.items():
    ax.plot(r['step_loss'][:500], label=name, lw=1.2)
ax.axhline(math.log(10), color='k', ls='--', lw=1, alpha=.5)
ax.text(5, math.log(10) + .08, 'ln(10) = 2.30: guessing level', fontsize=8)
ax.set_ylim(0, 6); ax.set_xlabel('training step (mini-batch)'); ax.set_ylabel('batch loss')
ax.set_title('Per-step training loss, first 500 steps (y-axis clipped at 6)'); ax.legend(fontsize=8)
plt.show()

### 8b. Epoch-level curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for name, r in results.items():
    e = range(1, len(r['train_loss']) + 1)
    ax[0].plot(e, r['train_loss'], marker='o', ms=3, label=name)
    ax[1].plot(e, r['test_acc'],   marker='o', ms=3, label=name)
ax[0].set_title('train loss'); ax[0].set_ylim(0, 3)
ax[1].set_title('test accuracy'); ax[1].set_ylim(0, 1)
for a in ax: a.set_xlabel('epoch')
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

### 8c. Summary

In [ ]:
rows = []
for name, r in results.items():
    if r['diverged']:
        rows.append(dict(init=name, epochs_completed=len(r['train_loss']), final_train_loss=np.nan,
                         final_test_acc=np.nan, verdict='diverged (NaN)'))
        continue
    ta = r['test_acc'][-1]
    verdict = 'stuck near chance' if ta < 0.2 else ('poor' if ta < 0.8 else 'ok')
    rows.append(dict(init=name, epochs_completed=len(r['train_loss']), final_train_loss=r['train_loss'][-1],
                     final_test_acc=ta, verdict=verdict))
display(pd.DataFrame(rows).set_index('init').round(4))

## 9. What to look for

- **zeros:** the loss stays at 2.30 and accuracy stays at 10%. With all-zero weights the ReLU outputs are all zero, so the gradients to every hidden layer are exactly zero and nothing there can ever change. Only the last bias learns (to predict class frequencies, which are equal here).
- **constant 0.05:** it does learn something, but far worse than a random init. Every neuron in a layer starts identical, receives identical gradients, and stays identical forever (see section 10). A 128-neuron layer behaves like one neuron.
- **tiny (std 0.001):** typically a long flat plateau near 2.30 (the signal is too weak for useful gradients), then maybe a slow take-off, or no take-off within 10 epochs.
- **huge (std 1.0):** starts at an enormous loss, with very jagged training and a possible blow-up (NaN). Even if it recovers, it ends up worse than a well-scaled init.
- **PyTorch default, Xavier, Kaiming:** all train smoothly, starting close to loss 2.30 and ending within a point or two of each other. In a shallow network like this one, the choice among the well-scaled inits barely matters. Section 4 showed that depth is where it starts to matter.

**Rule:** a good init (1) breaks symmetry with randomness, and (2) is scaled by `fan_in` so signals neither vanish nor explode.

## 10. Proof of the symmetry problem

If the symmetry argument is right, then after training with the constant init, *every neuron in a hidden layer should still have exactly the same weights*. Check it. Compare that with the Kaiming-initialized network, where the neurons should all be different.

In [ ]:
def neurons_identical(model):
    W1 = model.net[1].weight                     # (128, 784): one row per hidden neuron
    return torch.allclose(W1[0], W1[1], atol=1e-6), (W1[0] - W1[1]).abs().max().item()

for name in ['constant 0.05', 'Kaiming (He)']:
    same, diff = neurons_identical(results[name]['model'])
    print(f'{name:15s} neuron 0 == neuron 1 after training?  {same}   (max |difference| = {diff:.2e})')

W1 = results['constant 0.05']['model'].net[1].weight.detach().cpu()
print('distinct rows among the 128 hidden neurons (constant init):', torch.unique(W1.round(decimals=5), dim=0).shape[0])

Under the constant init, all 128 neurons of the first hidden layer remain identical copies of each other, so the layer is 128 neurons that all compute the same thing. Random initialization is what lets each neuron specialize in something different.

## Takeaways

1. **Never initialize all weights to the same value** (zeros included): the neurons stay identical and the network cannot use its width.
2. **Scale matters:** too small and the signal and gradients vanish; too large and they explode. Scale the random weights by `fan_in` (Xavier/Kaiming), not by a fixed number.
3. **Match the init to the activation:** Kaiming (He) for ReLU-type activations, Xavier (Glorot) for tanh/sigmoid.
4. In shallow networks PyTorch's default is fine; init matters more as networks get deeper.
5. **Quick sanity check:** before training, the loss should be close to `ln(num_classes)`. If it isn't, something is wrong with the init (or the data/loss).

## Exercises
1. Try `init_normal(0.01)`, `init_normal(0.05)`, `init_normal(0.1)`. Where is the sweet spot? How does it relate to `1/sqrt(784)`?
2. Re-run with `act=nn.Tanh` in `MLP`. Does Xavier now beat Kaiming?
3. Make the network deeper (`hidden=(128,)*8`) and repeat the comparison. Which inits break first?
4. Initialize only the *last* layer to zeros and everything else with Kaiming. Does training still work? Why?
5. Change the constant init to a mix: random weights in layer 1 only, constant in layers 2 and 3. Which symmetry survives?
6. Use `nn.init.orthogonal_` or `nn.init.trunc_normal_` and compare.